# TP 2 : Embeddings statiques et recherche d’information

In [2]:
from pathlib import Path
import pandas as pd
import spacy
import sklearn
from gensim.models import Word2Vec
import numpy as np
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


## 2. Chargement du corpus MIRACL-FR-Mini-30K

### 2.1 Localisation des fichiers

In [15]:

def trouver_fichier(nom_fichier):
    path = Path(nom_fichier)
    if path.exists():
        return path
    else:
        raise FileNotFoundError(f"Le fichier {nom_fichier} n'a pas été trouvé.")


In [16]:
chemin_corpus = trouver_fichier("corpus.jsonl")
chemin_requetes = trouver_fichier("queries.jsonl")
chemin_frWac = trouver_fichier("/Users/ghita_eddial/IR_S5/frWac_non_lem_no_postag_no_phrase_200_skip_cut100 2.bin")


### 2.2 Chargement des données

In [18]:
corpus = pd.read_json(chemin_corpus, lines=True)
requetes_df = pd.read_json(chemin_requetes, lines=True)

In [19]:
print('chemin_corpus: ', chemin_corpus)
print('Nombre de documents dans le corpus: ', len(corpus))
print('chemin_requetes: ', chemin_requetes)
print('Nombre de requêtes: ', len(requetes_df))

chemin_corpus:  corpus.jsonl
Nombre de documents dans le corpus:  30000
chemin_requetes:  queries.jsonl
Nombre de requêtes:  300


In [20]:
corpus.head()

,_id,title,text
0,1000019#8,Police provinciale de l'Ontario,Les services opérationnels sur le terrain sont...
1,100013#0,Jacques Offenbach,"Jacques Offenbach ( """" ) est un compositeur et..."
2,10002#36,Chimpanzé,Pour préserver la faune et lutter contre les m...
3,1000250#15,Industrie nucléaire,"Pour ces derniers métaux, la consommation de l..."
4,10004#1,Brigitte Le Brethon,Professeur agrégée d'économie et gestion dans ...


In [21]:
requetes_df.head()

,_id,text
0,fr-dev-0001,Qu'est-ce que c'est une famille patricienne?
1,fr-dev-0002,Pourquoi est-ce que Saint Patrick est célèbre?
2,fr-dev-0003,Pourquoi on a besoin de charbon?
3,fr-dev-0004,Pourquoi est-ce qu'on peut construire un ecovi...
4,fr-dev-0005,Comment sont créés les astéroïdes?


### 2.3 Vérification du format

In [25]:
print('_id' in corpus)
print('_id' in requetes_df)
print('text' in corpus)
print('text' in requetes_df)


True
True
True
True


In [45]:
def regroupement_title_text(df):
    if df['title'].notnull().any():
        df['texte_complet'] = df['title'] + " " + df['text']
    else:
        df['texte_complet'] = df['text']
    

In [46]:
regroupement_title_text(corpus)

In [47]:
corpus.head()

,_id,title,text,texte_complet
0,1000019#8,Police provinciale de l'Ontario,Les services opérationnels sur le terrain sont...,Police provinciale de l'Ontario Les services o...
1,100013#0,Jacques Offenbach,"Jacques Offenbach ( """" ) est un compositeur et...","Jacques Offenbach Jacques Offenbach ( """" ) est..."
2,10002#36,Chimpanzé,Pour préserver la faune et lutter contre les m...,Chimpanzé Pour préserver la faune et lutter co...
3,1000250#15,Industrie nucléaire,"Pour ces derniers métaux, la consommation de l...","Industrie nucléaire Pour ces derniers métaux, ..."
4,10004#1,Brigitte Le Brethon,Professeur agrégée d'économie et gestion dans ...,Brigitte Le Brethon Professeur agrégée d'écono...


In [51]:
print(requetes_df[['text']].isna().sum())
requetes_df[['text']].dropna(inplace=True)

text    0
dtype: int64


### Pourquoi est-il utile d’ajouter le titre au contenu du document avant sa représentation vectorielle ?
    Parce que sinon on aura besoin de faire le traitemnet sur le titre et le text chacun , donc on les concatene poour ne pas perdre de l'information et aussi pas perdre du temps. 

## 3. Prétraitement linguistique

In [52]:
nlp = spacy.blank("fr")

In [53]:
def extraire_tokens_contexte(doc):
    doc = nlp(doc)
    tokens = [token.text for token in doc if not token.is_punct and not token.is_space and token.is_alpha]
    return [token.lower() for token in tokens]

In [55]:
docs = nlp.pipe(corpus['texte_complet'].astype(str), batch_size=64)

corpus["tokens_contexte"] = [extraire_tokens_contexte(doc) for doc in docs if doc is not None]



In [56]:
corpus["tokens_contexte"][0:3]

0    [police, provinciale, de, ontario, les, servic...
1    [jacques, offenbach, jacques, offenbach, est, ...
2    [chimpanzé, pour, préserver, la, faune, et, lu...
Name: tokens_contexte, dtype: object

### Pourquoi les mots vides sont-ils conservés pendant l’apprentissage de Word2Vec et FastText ?
Parce que sinon on peut perdre le contexte , et le smots vides peuvent  apporter une info contextuelle. 

## 4. Entraînement d’un modèle Word2Vec local

### 4.1 Entraînement du modèle

In [59]:
word2vec_model = Word2Vec(
    sentences=corpus["tokens_contexte"],
    vector_size=100,
    window=5,
    min_count=3,
    workers=1,
    sg=1,
    negative=10,
    epochs=10,
    seed=42
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [ ]:
vecteurs_w2v = word2vec_model.wv
print('La taille du vocabulaire :', len(vecteurs_w2v))


La taille du vocabulaire : 35977


In [72]:
vecteurs_w2v.vectors.shape

(35977, 100)

In [73]:
del word2vec_model

In [75]:
import gc
gc.collect()

487

### Questions :

####  Que signifie le paramètre sg=1 ?
sg=1 signifie que Word2Vec utilise l’architecture Skip-gram.
####  Pourquoi le vocabulaire appris contient-il moins de mots que l’ensemble des formes rencontrées dans le corpus ?
Le vocabulaire appris contient moins de mots que toutes les formes rencontrées dans le corpus à cause de min_count=3 : les mots apparaissant moins de 3 fois sont ignorés lors de l’apprentissage.
#### Pourquoi l’augmentation du nombre d’époques ne garantit-elle pas toujours une amélioration du modèle ?
Augmenter le nombre d’époques ne garantit pas toujours une amélioration, car après un certain point le modèle peut sur-apprendre les données du corpus.

###  4.2 Voisins sémantiques

In [82]:
def voisins_si_possible(mot, vecteurs_w2v):
    if mot not in vecteurs_w2v.key_to_index:
        return f"Le mot '{mot}' est absent du vocabulaire."
    
    return vecteurs_w2v.most_similar(mot, topn=5)

In [79]:
mots = [
    "président",
    "gouvernement",
    "ministre",
    "politique",
    "élection",
    "parlement",
    "parti",
    "député",
    "république",
    "état"
]


In [80]:
mots_presents = [
    mot for mot in mots
    if mot in vecteurs_w2v.key_to_index
][:5]

In [83]:
for mot in mots_presents:
    print(f"\nMot : {mot}")
    
    voisins = voisins_si_possible(mot, vecteurs_w2v)
    
    for voisin, similarite in voisins:
        print(f"{voisin} : {similarite:.4f}")


Mot : président
présidence : 0.7034
obama : 0.6781
barack : 0.6777
élu : 0.6756
secrétaire : 0.6723

Mot : gouvernement
ministre : 0.6697
portefeuille : 0.6615
roca : 0.6574
quisling : 0.6548
ministres : 0.6533

Mot : ministre
ministère : 0.7038
secrétaire : 0.6972
portefeuille : 0.6827
cabinet : 0.6739
finances : 0.6731

Mot : politique
personnalité : 0.7192
gaulliste : 0.6836
démocrate : 0.6744
démocratie : 0.6688
dénonciation : 0.6661

Mot : élection
présidentielle : 0.8529
élections : 0.7505
électeurs : 0.7257
législatives : 0.7253
députés : 0.7182


### Question, Les mots les plus proches sont-ils nécessairement des synonymes ? Expliquez la nature de la proximité apprise par Word2Vec.

Non. Word2Vec rapproche surtout les mots utilisés dans des contextes similaires, pas forcément les synonymes.

## 5. Entraînement d’un modèle FastText local

In [84]:
from gensim.models import FastText

modele_ft = FastText(
    sentences=corpus["tokens_contexte"],
    vector_size=100,
    window=5,
    min_count=3,
    workers=1,
    sg=1,
    negative=10,
    epochs=10,
    seed=42,
    min_n=3,
    max_n=6,
    bucket=200000
)

vecteurs_ft = modele_ft.wv

print("Taille du vocabulaire :", len(vecteurs_ft.key_to_index))

print("Dimension des vecteurs :", vecteurs_ft.vector_size)

del modele_ft
gc.collect()

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Taille du vocabulaire : 35977
Dimension des vecteurs : 100


736

### 5.1 Variantes morphologiques

In [85]:
mots = ["élection", "élections", "électoral", "électorale"]

for mot in mots:
    print(f"\nMot : {mot}")

    # 1. Vérifier la présence dans le vocabulaire explicite
    print("Dans le vocabulaire explicite :", mot in vecteurs_ft.key_to_index)

    try:
        # 2. Récupérer ou construire le vecteur
        vecteur = vecteurs_ft.get_vector(mot)

        # 3. Afficher sa norme
        print("Norme :", np.linalg.norm(vecteur))

        # 4. Afficher les 5 voisins les plus proches
        voisins = vecteurs_ft.most_similar(mot, topn=5)

        print("Voisins :")
        for voisin, similarite in voisins:
            print(f"{voisin} : {similarite:.4f}")

    except KeyError:
        print("Aucun vecteur disponible.")


Mot : élection
Dans le vocabulaire explicite : True
Norme : 5.5313005
Voisins :
réélection : 0.9590
électorat : 0.8228
élections : 0.8156
électorale : 0.7921
législative : 0.7749

Mot : élections
Dans le vocabulaire explicite : True
Norme : 5.775189
Voisins :
électorales : 0.8619
elections : 0.8561
législatives : 0.8557
sélections : 0.8217
législations : 0.8163

Mot : électoral
Dans le vocabulaire explicite : True
Norme : 4.7872415
Voisins :
électorat : 0.9344
électorale : 0.9066
électoraux : 0.8966
électorales : 0.8539
législatif : 0.8193

Mot : électorale
Dans le vocabulaire explicite : True
Norme : 4.4338117
Voisins :
électoral : 0.9066
électorat : 0.8893
électorales : 0.8591
électoraux : 0.8477
législative : 0.8410


## Question
Quelle différence existe-t-il entre l’appartenance au vocabulaire explicite et la disponibilité
d’un vecteur FastText ?

L’appartenance au vocabulaire explicite signifie que le mot complet a été appris pendant l’entraînement.
Avec FastText, un mot absent de ce vocabulaire peut quand même avoir un vecteur, construit à partir de ses sous-mots (n-grams de caractères).

## 6. Comparaison de Word2Vec et FastText sur les mots rares

In [86]:
from collections import Counter

frequences_corpus = Counter(
    mot
    for tokens_document in corpus["tokens_contexte"]
    for mot in tokens_document
)

In [87]:
mots_rares = [
    "université",
    "universitaire",
    "interuniversitaires",
    "scolarisée"
]

for mot in mots_rares:
    print(f"\n===== {mot} =====")

    # 1. Fréquence réelle dans le corpus
    print("Fréquence :", frequences_corpus[mot])

    # 2. Présence dans le vocabulaire Word2Vec
    present_w2v = mot in vecteurs_w2v.key_to_index
    print("Présent dans Word2Vec :", present_w2v)

    # 3. Tenter de récupérer le vecteur Word2Vec
    try:
        vecteur_w2v = vecteurs_w2v.get_vector(mot)
        print("Vecteur Word2Vec disponible : Oui")
    except KeyError:
        print("Vecteur Word2Vec disponible : Non")

    # 4. Présence dans le vocabulaire explicite FastText
    present_ft = mot in vecteurs_ft.key_to_index
    print("Présent dans le vocabulaire explicite FastText :", present_ft)

    # 5. Tenter de récupérer le vecteur FastText
    try:
        vecteur_ft = vecteurs_ft.get_vector(mot)
        print("Vecteur FastText disponible : Oui")

        # 6. Norme du vecteur
        print("Norme :", np.linalg.norm(vecteur_ft))

        # Voisins
        print("5 voisins FastText :")
        voisins = vecteurs_ft.most_similar(mot, topn=5)

        for voisin, similarite in voisins:
            print(f"{voisin} : {similarite:.4f}")

    except KeyError:
        print("Vecteur FastText disponible : Non")


===== université =====
Fréquence : 677
Présent dans Word2Vec : True
Vecteur Word2Vec disponible : Oui
Présent dans le vocabulaire explicite FastText : True
Vecteur FastText disponible : Oui
Norme : 5.133675
5 voisins FastText :
university : 0.8975
universités : 0.8681
universidad : 0.8630
universitaire : 0.8574
universiade : 0.8560

===== universitaire =====
Fréquence : 91
Présent dans Word2Vec : True
Vecteur Word2Vec disponible : Oui
Présent dans le vocabulaire explicite FastText : True
Vecteur FastText disponible : Oui
Norme : 4.124906
5 voisins FastText :
interuniversitaire : 0.9439
universitaires : 0.8704
universidad : 0.8695
university : 0.8643
université : 0.8574

===== interuniversitaires =====
Fréquence : 1
Présent dans Word2Vec : False
Vecteur Word2Vec disponible : Non
Présent dans le vocabulaire explicite FastText : False
Vecteur FastText disponible : Oui
Norme : 3.3618982
5 voisins FastText :
universitaires : 0.9492
interuniversitaire : 0.9036
universités : 0.8452
universit

### Réponses aux questions

1. **Fréquence et `min_count`**  
Avec `min_count=3`, un mot doit apparaître au moins 3 fois pour être dans le vocabulaire explicite.

2. **Mot absent avec FastText**  
FastText peut créer un vecteur grâce aux sous-mots du mot.

3. **Qualité du vecteur FastText**  
Non. Un vecteur peut exister sans être sémantiquement très pertinent.

## 7. Préparation spécifique à la recherche d’information

In [88]:
def supprimer_mots_vides(tokens):
    return [
        mot for mot in tokens
        if not nlp.vocab[mot].is_stop
    ]

corpus["tokens_ri"] = corpus["tokens_contexte"].apply(supprimer_mots_vides)

In [89]:
corpus[
    ["texte_complet", "tokens_contexte", "tokens_ri"]
].head(3)

,texte_complet,tokens_contexte,tokens_ri
0,Police provinciale de l'Ontario Les services o...,"[police, provinciale, de, ontario, les, servic...","[police, provinciale, ontario, services, opéra..."
1,"Jacques Offenbach Jacques Offenbach ( """" ) est...","[jacques, offenbach, jacques, offenbach, est, ...","[jacques, offenbach, jacques, offenbach, compo..."
2,Chimpanzé Pour préserver la faune et lutter co...,"[chimpanzé, pour, préserver, la, faune, et, lu...","[chimpanzé, préserver, faune, lutter, contre, ..."


On conserve `tokens_contexte` pour entraîner Word2Vec/FastText avec le contexte complet, y compris les mots outils.

On conserve `tokens_ri` pour la recherche d’information, afin de réduire l’influence des mots vides peu informatifs.

## 8. Utilisation du modèle Word2Vec français frWaC

In [90]:
from pathlib import Path
from gensim.models import KeyedVectors

chemin_frwac = Path(
    "frWac_non_lem_no_postag_no_phrase_200_skip_cut100.bin"
)

if chemin_frwac.exists():
    vecteurs_frwac = KeyedVectors.load_word2vec_format(
        chemin_frwac,
        binary=True,
        unicode_errors="ignore"
    )

    print("Taille du vocabulaire :", len(vecteurs_frwac.key_to_index))
    print("Dimension des vecteurs :", vecteurs_frwac.vector_size)

else:
    vecteurs_frwac = None
    print("Le fichier frWaC est absent. On poursuit avec les modèles locaux.")

Le fichier frWaC est absent. On poursuit avec les modèles locaux.


### 8.1 Comparaison avec Word2Vec local

In [91]:
mots_thematiques = [
    "président",
    "gouvernement",
    "ministre",
    "politique",
    "élection",
    "parlement",
    "parti",
    "député",
    "république",
    "état"
]

for mot in mots_thematiques:
    print(f"\n===== {mot} =====")

    if mot in vecteurs_w2v.key_to_index:
        print("Word2Vec local :")
        for voisin, score in vecteurs_w2v.most_similar(mot, topn=5):
            print(f"{voisin} : {score:.4f}")

    if vecteurs_frwac is not None and mot in vecteurs_frwac.key_to_index:
        print("\nfrWaC :")
        for voisin, score in vecteurs_frwac.most_similar(mot, topn=5):
            print(f"{voisin} : {score:.4f}")


===== président =====
Word2Vec local :
présidence : 0.7034
obama : 0.6781
barack : 0.6777
élu : 0.6756
secrétaire : 0.6723

===== gouvernement =====
Word2Vec local :
ministre : 0.6697
portefeuille : 0.6615
roca : 0.6574
quisling : 0.6548
ministres : 0.6533

===== ministre =====
Word2Vec local :
ministère : 0.7038
secrétaire : 0.6972
portefeuille : 0.6827
cabinet : 0.6739
finances : 0.6731

===== politique =====
Word2Vec local :
personnalité : 0.7192
gaulliste : 0.6836
démocrate : 0.6744
démocratie : 0.6688
dénonciation : 0.6661

===== élection =====
Word2Vec local :
présidentielle : 0.8529
élections : 0.7505
électeurs : 0.7257
législatives : 0.7253
députés : 0.7182

===== parlement =====
Word2Vec local :
législature : 0.7139
storting : 0.7115
élisent : 0.6930
siéger : 0.6887
bundestag : 0.6839

===== parti =====
Word2Vec local :
libéral : 0.8341
démocrate : 0.8101
travailliste : 0.8060
socialiste : 0.7945
communiste : 0.7848

===== député =====
Word2Vec local :
réélu : 0.8228
élu : 0.

### Réponses

1. **Quel modèle donne les voisins les plus cohérents ?**  
En général, frWaC donne des voisins plus cohérents et plus stables.

2. **Pourquoi frWaC est-il plus stable ?**  
Parce qu’il a été entraîné sur un corpus beaucoup plus grand et plus varié que MIRACL-FR-Mini-30K.

3. **Quand le modèle local peut-il être utile ?**  
Il peut mieux refléter les associations spécifiques au corpus MIRACL, donc être plus adapté à son domaine.

## 9. Construction des vecteurs de documents

In [92]:
import numpy as np

def vecteur_moyen(tokens, kv):
    vecteurs = []

    for mot in tokens:
        try:
            vecteurs.append(kv.get_vector(mot))
        except KeyError:
            pass

    if len(vecteurs) == 0:
        return np.zeros(kv.vector_size)

    return np.mean(vecteurs, axis=0)

In [93]:
def construire_matrice_documents(serie_tokens, kv):
    return np.vstack([
        vecteur_moyen(tokens, kv)
        for tokens in serie_tokens
    ])

In [94]:
matrice_w2v = construire_matrice_documents(
    corpus["tokens_ri"],
    vecteurs_w2v
)

matrice_ft = construire_matrice_documents(
    corpus["tokens_ri"],
    vecteurs_ft
)

In [95]:
if vecteurs_frwac is not None:
    matrice_frwac = construire_matrice_documents(
        corpus["tokens_ri"],
        vecteurs_frwac
    )

In [96]:
modeles = {
    "Word2Vec local": vecteurs_w2v,
    "FastText local": vecteurs_ft
}

if vecteurs_frwac is not None:
    modeles["frWaC"] = vecteurs_frwac

In [97]:
matrices = {
    "Word2Vec local": matrice_w2v,
    "FastText local": matrice_ft
}

if vecteurs_frwac is not None:
    matrices["frWaC"] = matrice_frwac

In [98]:
for nom, matrice in matrices.items():
    print(nom, ":", matrice.shape)

Word2Vec local : (30000, 100)
FastText local : (30000, 100)


### Questions

1. **Lignes et colonnes**  
Chaque ligne représente un document et chaque colonne une dimension du vecteur.

2. **Pourquoi frWaC a plus de colonnes ?**  
Parce que ses vecteurs n'ont pas la même dimension que ceux des modèles locaux.

3. **Qu'est-ce qui est perdu avec la moyenne ?**  
La moyenne perd notamment l'ordre des mots et une partie du contexte et de la structure de la phrase.

## 10. Recherche sémantique dans MIRACL

### 10.1 Préparation des requêtes

In [100]:
def preparer_requete(texte):
    doc = nlp(str(texte).lower())

    tokens = [
        token.text
        for token in doc
        if token.is_alpha and not token.is_stop
    ]

    return tokens

In [101]:
preparer_requete("Quel est le rôle du président de la République ?")

['rôle', 'président', 'république']

### Question

Il faut appliquer le même prétraitement aux documents et aux requêtes pour qu'ils soient représentés de manière cohérente et puissent être comparés correctement.

### 10.2 Fonction de recherche

In [102]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def rechercher(requete, matrice_documents, kv, top_k=5):

    # 1. Préparer la requête
    tokens_requete = preparer_requete(requete)

    # 2. Calculer son vecteur moyen
    vecteur_requete = vecteur_moyen(tokens_requete, kv)

    # 3. Vérifier si le vecteur est nul
    if np.linalg.norm(vecteur_requete) == 0:
        print("Aucun token de la requête ne peut être représenté.")
        return None

    # 4. Calculer les similarités cosinus
    scores = cosine_similarity(
        vecteur_requete.reshape(1, -1),
        matrice_documents
    )[0]

    # 5 et 6. Trier et garder les top_k meilleurs
    indices = scores.argsort()[::-1][:top_k]

    # 7. Retourner identifiant, score et extrait
    resultats = corpus.iloc[indices][["_id", "texte_complet"]].copy()

    resultats["score"] = scores[indices]
    resultats["extrait"] = resultats["texte_complet"].str.slice(0, 200)

    return resultats[["_id", "score", "extrait"]]

In [103]:
rechercher(
    "président de la république",
    matrice_w2v,
    vecteurs_w2v
)

,_id,score,extrait
26997,784#17,0.855482,République du Congo En 2002 est adopté une nou...
15061,3211545#6,0.851566,Kazimierz Sabbat Désigné le comme futur succes...
11572,2452827#8,0.830229,"Michel Aoun En septembre 1988, le président so..."
8468,165022#9,0.829973,"Fausto Bertinotti Du 8 au , Fausto Bertinotti,..."
13069,283580#3,0.827183,"Kendalc'h En , la Charte culturelle bretonne, ..."


### Question

Avec Word2Vec, une requête obtient un vecteur nul lorsqu'aucun de ses tokens n'est présent dans le vocabulaire du modèle.

### 10.3 Test sur des requêtes réelles

In [104]:
requetes_test = requetes_df[
    requetes_df["texte_complet"].notna()
    & (requetes_df["texte_complet"].str.strip() != "")
].head(3)

In [105]:
requetes = {
    row["_id"]: row["texte_complet"]
    for _, row in requetes_test.iterrows()
}

requetes

{'fr-dev-0001': "Qu'est-ce que c'est une famille patricienne?",
 'fr-dev-0002': 'Pourquoi est-ce que Saint Patrick est célèbre?',
 'fr-dev-0003': 'Pourquoi on a besoin de charbon?'}

In [106]:
classements = {}
for id_requete, texte_requete in requetes.items():

    print(f"\n==============================")
    print(f"Requête {id_requete} : {texte_requete}")
    print(f"==============================")

    classements[id_requete] = {}

    for nom_modele in modeles:

        print(f"\n--- {nom_modele} ---")

        resultats = rechercher(
            texte_requete,
            matrices[nom_modele],
            modeles[nom_modele],
            top_k=5
        )

        classements[id_requete][nom_modele] = resultats

        if resultats is not None:
            display(resultats)


Requête fr-dev-0001 : Qu'est-ce que c'est une famille patricienne?

--- Word2Vec local ---


,_id,score,extrait
1845,11574018#0,0.982109,Famille Patry La famille Patry est une famille...
14386,30973#11,0.973139,"Asphodelaceae En , la famille des ""Asphodelace..."
21326,4955330#0,0.959780,Lamprochernetinae Les Lamprochernetinae sont u...
20952,4869627#21,0.930797,Familles d'Abadie Famille béarnaise éteinte au .
29094,912285#0,0.921078,Famille Dubois de La Patellière La famille Dub...



--- FastText local ---


,_id,score,extrait
1845,11574018#0,0.952760,Famille Patry La famille Patry est une famille...
20952,4869627#21,0.916452,Familles d'Abadie Famille béarnaise éteinte au .
14386,30973#11,0.879270,"Asphodelaceae En , la famille des ""Asphodelace..."
29094,912285#0,0.864503,Famille Dubois de La Patellière La famille Dub...
17321,3715554#2,0.855397,Famille Pomare Les souverains sont :



Requête fr-dev-0002 : Pourquoi est-ce que Saint Patrick est célèbre?

--- Word2Vec local ---


...

,_id,score,extrait
26782,7767661#2,0.858034,Église Saint-Valentin de Guilers Le Saint patr...
6184,1413550#26,0.855225,"Patrick d'Irlande Pour les catholiques, saint ..."
25116,6726842#9,0.853400,Église San Peyre (Stroppo) Au cours des premiè...
4510,12876#34,0.835169,"Guipavas Saint Thudon est parfois, mais cela r..."
23651,6292323#0,0.828336,Tropez de Pise Saint Tropez de Pise (ou cheval...



Requête fr-dev-0003 : Pourquoi on a besoin de charbon?

--- Word2Vec local ---


,_id,score,extrait
22129,5329820#45,0.830467,"Charbon Le charbon couvrait 26,8 % des besoins..."
22128,5329820#1,0.820409,"Charbon Couvrant 26,8 % des besoins énergétiqu..."
3395,12336315#6,0.807036,Sortie des combustibles fossiles Les partisans...
13102,284062#81,0.795333,Histoire de la production de l'acier Il lui vi...
26947,78221#20,0.785372,Houille L'utilisation du charbon dans les cent...



--- FastText local ---


,_id,score,extrait
22129,5329820#45,0.823593,"Charbon Le charbon couvrait 26,8 % des besoins..."
22128,5329820#1,0.810723,"Charbon Couvrant 26,8 % des besoins énergétiqu..."
13102,284062#81,0.804801,Histoire de la production de l'acier Il lui vi...
12758,2777010#15,0.796849,Champignonnière Certains champignons n'ont pas...
3395,12336315#6,0.788713,Sortie des combustibles fossiles Les partisans...


In [107]:
modeles = {
    "Word2Vec local": vecteurs_w2v,
    "FastText local": vecteurs_ft
}

matrices = {
    "Word2Vec local": matrice_w2v,
    "FastText local": matrice_ft
}

In [108]:
if vecteurs_frwac is not None:
    modeles["frWaC"] = vecteurs_frwac
    matrices["frWaC"] = matrice_frwac

In [109]:
classements[id_requete]["Word2Vec local"]

,_id,score,extrait
22129,5329820#45,0.830467,"Charbon Le charbon couvrait 26,8 % des besoins..."
22128,5329820#1,0.820409,"Charbon Couvrant 26,8 % des besoins énergétiqu..."
3395,12336315#6,0.807036,Sortie des combustibles fossiles Les partisans...
13102,284062#81,0.795333,Histoire de la production de l'acier Il lui vi...
26947,78221#20,0.785372,Houille L'utilisation du charbon dans les cent...


### 10.4 Analyse des classements

### 10.4 Analyse des classements

Pour chaque requête, on compare les premiers documents retournés par Word2Vec local, FastText local et frWaC en observant :
- si le premier document répond bien à la requête ;
- si les concepts principaux de la requête sont présents ;
- les différences entre les résultats des modèles ;
- les éventuels documents peu pertinents.

### Questions

1. **Pourquoi deux modèles peuvent-ils produire des classements différents ?**  
Parce qu’ils ont été entraînés différemment et n’apprennent pas exactement les mêmes représentations des mots.

2. **Un score cosinus élevé garantit-il la pertinence ?**  
Non. Il indique seulement une forte proximité vectorielle, pas forcément une vraie pertinence pour la requête.

3. **Pourquoi ne pas comparer directement les scores cosinus de deux modèles ?**  
Parce que chaque modèle possède son propre espace vectoriel et sa propre échelle de similarité.

4. **Quel modèle donne les résultats les plus cohérents ?**  
Cela dépend des documents retournés. En général, frWaC peut être plus stable grâce à son entraînement sur un corpus plus large, tandis que FastText peut mieux gérer les mots rares ou inconnus.